# 6장 2강: 허깅페이스 모델 실습
## 2. 트랜스포머 아키텍처별 모델 실습


### 2.2 인코더 모델 활용: KoBERT로 감정 분류

In [2]:
# 토크나이저 및 모델 로드
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# KoBERT 토크나이저와 모델 로드
tokenizer = AutoTokenizer.from_pretrained("monologg/kobert", trust_remote_code=True)
model = AutoModelForSequenceClassification.from_pretrained("rkdaldus/ko-sent5-classification")

# 사용자 입력 텍스트 감정 분석
#text = "오늘 정말 행복해!"
text = "너무 무서워!"
inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
print("inputs:", inputs)
with torch.no_grad():
    outputs = model(**inputs)
predicted_label = torch.argmax(outputs.logits, dim=1).item()

# 감정 레이블 정의
emotion_labels = {
    0: ("Angry", "😡"),
    1: ("Fear", "😨"),
    2: ("Happy", "😊"),
    3: ("Tender", "🥰"),
    4: ("Sad", "😢")
}

# 예측된 감정 출력
print(f"예측된 감정: {emotion_labels[predicted_label][0]} {emotion_labels[predicted_label][1]}")

c:\Users\jihyu\Documents\AX_study\04.Machine Learning\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jihyu\.cache\huggingface\hub\models--rkdaldus--ko-sent5-classification. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9943.61it/s]
[transf

inputs: {'input_ids': tensor([[   2, 1458, 2095, 6553, 7018,    5,    3]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1]])}
예측된 감정: Fear 😨


### 2.3 인코더-디코더 모델 활용: KoBART로 뉴스 요약

In [1]:
import torch
from transformers import PreTrainedTokenizerFast
from transformers import BartForConditionalGeneration

tokenizer = PreTrainedTokenizerFast.from_pretrained('gogamza/kobart-summarization')
model = BartForConditionalGeneration.from_pretrained('gogamza/kobart-summarization')

text = "과거를 떠올려보자. 방송을 보던 우리의 모습을. 독보적인 매체는 TV였다. 온 가족이 둘러앉아 TV를 봤다. 간혹 가족들끼리 뉴스와 드라마, 예능 프로그램을 둘러싸고 리모컨 쟁탈전이 벌어지기도  했다. 각자 선호하는 프로그램을 ‘본방’으로 보기 위한 싸움이었다. TV가 한 대인지 두 대인지 여부도 그래서 중요했다. 지금은 어떤가. ‘안방극장’이라는 말은 옛말이 됐다. TV가 없는 집도 많다. 미디어의 혜 택을 누릴 수 있는 방법은 늘어났다. 각자의 방에서 각자의 휴대폰으로, 노트북으로, 태블릿으로 콘텐츠 를 즐긴다."

raw_input_ids = tokenizer.encode(text)
input_ids = [tokenizer.bos_token_id] + raw_input_ids + [tokenizer.eos_token_id]

summary_ids = model.generate(torch.tensor([input_ids]))
tokenizer.decode(summary_ids.squeeze().tolist(), skip_special_tokens=True)

c:\Users\jihyu\Documents\AX_study\04.Machine Learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\jihyu\Documents\AX_study\04.Machine Learning\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jihyu\.cache\huggingface\hub\models--gogamza--kobart-summarization. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to ru

'TV가 없는 집도 많아지고 미디어의 혜 택을 누릴 수 있는 방법은 늘어났다.'

### 2.4 디코더 모델 활용: Gemma로 대화형 텍스트 생성

- 디코더 전용 구조는 문장 생성(NLG) 태스크를 수행합니다. 허깅페이스의 상위 인터페이스인 pipeline 구조 내부에서 가속기를 활용할 수 있도록 device_map="auto" 옵션을 주입합니다. 가상환경 내에 설치된 accelerate 패키지가 각 OS 운영체제 환경을 판별하여 최적의 가속 레이어(CUDA 또는 MPS)를 자동 매핑합니다.

In [4]:
# 교안 코딩 : 
from transformers import pipeline
import torch

# Gemma 인스트럭션 모델 식별자 설정
gemma_identifier = "google/gemma-2b-it"

# 텍스트 생성 파이프라인 구축 (bfloat16 연산 및 자원 자동 배치 적용)
gemma_generator = pipeline(
    "text-generation",
    model=gemma_identifier,
    dtype=torch.bfloat16,
    device_map="auto" # accelerate를 통해 OS별 하드웨어(CUDA/MPS) 최적 자동 할당
)

# 대화형 프롬프트 구조 정의
user_dialogue = [
    {"role": "user", "content": "내 이름은 무엇이지?"}
]

# 답변 생성
outputs = gemma_generator(user_dialogue, max_new_tokens=150)
print(outputs)


# 최종 답변 추출 및 출력
ai_reply = outputs[0]["generated_text"][-1]["content"]
print("--- Gemma 답변 ---")
print(f"Gemma의 답변:\n{ai_reply}")

Loading weights: 100%|██████████| 164/164 [00:00<00:00, 26522.69it/s]
Some parameters are on the meta device because they were offloaded to the disk and cpu.
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': [{'role': 'user', 'content': '내 이름은 무엇이지?'}, {'role': 'assistant', 'content': '나는 알 수 없는 정보이므로, 이름은 없습니다.'}]}]
--- Gemma 답변 ---
Gemma의 답변:
나는 알 수 없는 정보이므로, 이름은 없습니다.
